In [1]:
import os, pickle
import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt
from py_util_dx.py_utils import setProjectPath

In [ ]:
mpl.rcParams['font.family'] = 'helvetica'   # Global font family
mpl.rcParams['font.size'] = 12              # Global font size 
full_width = 8.5
half_width = full_width/2

In [ ]:
projectPath, mainResultsPath = setProjectPath()

dataset_name = 'MDTB'                       # or Demand

surface_helpers_dir = os.path.join(projectPath, 'surface_helpers')
resultsPath = os.path.join(mainResultsPath, 'START_A2_smooth_decomposition', dataset_name)

PKL_output = os.path.join(resultsPath, 'output.pkl')
with open(PKL_output, 'rb') as pf:
    output = pickle.load(pf)

In [ ]:
PKL_vis_output = os.path.join(resultsPath, 'vis_output.pkl')
vis_output = {}

In [ ]:
smooth_kernels = list(output.keys())
ROIs = list(output['orig'].keys())

v_s_over_gs = {}

for roi in ROIs:
    v_s_over_gs[roi] = []
    
    for smooth_kernel in smooth_kernels:
        v_s_over_gs[roi].append(output[smooth_kernel][roi][0, 1] / (output[smooth_kernel][roi][0, 0] + output[smooth_kernel][roi][0, 1]))

    v_s_over_gs[roi] = np.array(v_s_over_gs[roi])

vis_output['v_s_over_gs'] = v_s_over_gs

In [ ]:
colors = [(1, 150/255, 0), (171/255, 219/255, 227/255), (51/255, 153/255, 102/255), (135/255, 62/255, 35/255)]

plt.figure()

for roiI in np.arange(len(ROIs)):
    plt.scatter(0, v_s_over_gs[ROIs[roiI]][0], color=colors[roiI], marker='o')
    plt.plot(np.arange(1, len(ROIs)), v_s_over_gs[ROIs[roiI]][1:], color=colors[roiI], linestyle='-', label=ROIs[roiI])

plt.hlines(0, xmin=0, xmax=len(smooth_kernels)-1, linestyles='--', colors=(150/255, 150/255, 150/255))
plt.hlines(1, xmin=0, xmax=len(smooth_kernels)-1, linestyles='--', colors=(150/255, 150/255, 150/255))
plt.text(0, 0.05, 'group')
plt.text(0, 0.95, 'individual')

xlabels = smooth_kernels

plt.xticks(np.arange(len(smooth_kernels)), xlabels)
plt.title('$V_s/(V_s+V_g)$', fontsize=12)

plt.legend(frameon=False, loc='right')
plt.ylim([-0.1, 1.1])

plt.savefig(os.path.join(resultsPath, f'v_s_over_gs.jpg'), dpi=500, format='jpeg')

with open(PKL_vis_output, 'wb') as pk:
    pickle.dump(vis_output, pk)

plt.show()